# RepoMem — reproduce the benchmark and the paper's results

Companion notebook for **"RepoMem: Measuring and Repairing the Working-Memory Bottleneck in Small Gemma Code Agents"** (Gemma 4 Developer Agent — Paper Track).

This notebook:
1. regenerates the **RepoMem** benchmark (1,554 well-posed items) from the competition code graphs — the items themselves are Competition Data and are not redistributed, so they are rebuilt here;
2. checks its **well-posedness** and composition (distance buckets, prompts beyond the 32K window);
3. runs the **PFK-agent graph retrieval** offline (no GPU needed): retrieval success, buffer size vs. full prompt, and robustness to missing graph edges;
4. renders the paper's **result tables and figures** from the released aggregate results;
5. (optional) evaluates a Gemma 4 model with vLLM if a GPU and the model are attached.

**Inputs:** attach the `gemma-4-developer-agent` competition data. All files derived from it are written to `/tmp` only, so no Competition Data appears in this notebook's public outputs. Code: https://github.com/leomaxuz/repomem (Apache-2.0).

In [ ]:
import os, sys, glob, json, re, shutil, subprocess, collections, statistics as st
REPO_URL = "https://github.com/leomaxuz/repomem"
REPO = os.environ.get("REPOMEM_DIR", "/tmp/repomem")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
CODE = os.path.join(REPO, "code", "repomem")
sys.path.insert(0, CODE)
print("code:", CODE, sorted(os.listdir(CODE)))

## 1. Locate the competition code graphs

In [ ]:
# The competition ships 256 graph files: 129 task-named + 127 commit-named hard links of the same graphs.
# RepoMem uses the 127 unique commit-named graphs (<repo>_<40-hex sha>.json).
GRAPHS_IN = os.environ.get("REPOMEM_GRAPHS")
if not GRAPHS_IN:
    cands = [d for d in glob.glob("/kaggle/input/**/graphs", recursive=True) if os.path.isdir(d)]
    assert cands, "Attach the gemma-4-developer-agent competition data as an input."
    GRAPHS_IN = cands[0]
GRAPHS = "/tmp/repomem_graphs"   # outside /kaggle/working: nothing derived from Competition Data becomes a public output
os.makedirs(GRAPHS, exist_ok=True)
pat = re.compile(r"^[a-z]+_[0-9a-f]{40}\.json$")
for f in sorted(os.listdir(GRAPHS_IN)):
    if pat.match(f) and not os.path.exists(os.path.join(GRAPHS, f)):
        try:
            os.symlink(os.path.join(GRAPHS_IN, f), os.path.join(GRAPHS, f))
        except OSError:
            shutil.copy(os.path.join(GRAPHS_IN, f), os.path.join(GRAPHS, f))
print(len(os.listdir(GRAPHS)), "unique commit-named graphs (paper: 127)")

## 2. Regenerate RepoMem (main set: `--per-repo 20 --seed 1`)

In [ ]:
TASKS = os.path.join(os.path.dirname(GRAPHS), "repomem_tasks.jsonl")
subprocess.run([sys.executable, os.path.join(CODE, "generate.py"), "--graphs", GRAPHS,
                "--out", TASKS, "--per-repo", "20", "--seed", "1"], check=True,
               stdout=subprocess.DEVNULL)
items = [json.loads(l) for l in open(TASKS)]
B = [(0, 512), (512, 2000), (2000, 8000), (8000, 20000), (20000, 10**9)]
blab = ["0-512", "512-2K", "2K-8K", "8K-20K", "20K+"]
print("items:", len(items), "(paper: 1,554 well-posed; 97 ill-posed chain items filtered)")
print("by kind:", dict(collections.Counter(i["kind"] for i in items)), "(paper: defn 635, chain 411, caller 508)")
print("by distance:", {l: sum(lo <= i["distance_tokens"] < hi for i in items) for l, (lo, hi) in zip(blab, B)},
      "(paper: 369 / 360 / 501 / 283 / 41)")
approx = lambda s: len(s) // 4
print("prompts > ~32K tokens (approx. 4 chars/token):", sum(approx(i["prompt"]) > 32768 for i in items),
      "(paper, exact vLLM tokenizer: 42)")

## 3. Well-posedness checks

In [ ]:
from pfk_agent import load_graph, short
gcache = {}
def G(repo, drop=0.0):
    k = (repo, drop)
    if k not in gcache:
        gcache[k] = load_graph(os.path.join(GRAPHS, repo + ".json"), drop, 1)
    return gcache[k]

bad = collections.Counter()
for it in items:
    nodes, calls, callers = G(it["repo"])
    m = it["meta"]
    assert m["question"] in it["prompt"], "question must be stored verbatim"
    if it["kind"] == "chain":
        ch = m["chain"]
        for a, b in zip(ch, ch[1:]):
            if short(b) + "(" not in nodes[a]["text"]: bad["chain: call not textual"] += 1
    if it["kind"] == "caller":
        if len(callers.get(m["target"], [])) != 1: bad["caller: target not single-caller"] += 1
        if short(m["target"]) + "(" not in nodes[m["caller"]]["text"]: bad["caller: call not textual"] += 1
print("violations:", dict(bad) or "none")

## 4. PFK-agent retrieval, offline (no model needed)

In [ ]:
from pfk_agent import retrieve, func_block
def retrieval_stats(drop=0.0):
    ok, buf, full = 0, [], []
    for it in items:
        nodes, calls, callers = G(it["repo"], drop)
        ids, good = retrieve(it, nodes, calls, callers)
        ok += good
        buf.append(sum(len(func_block(nodes[i])) for i in ids if i in nodes) // 4)
        full.append(len(it["prompt"]) // 4)
    return ok / len(items), st.mean(buf), st.mean(full)
for drop in (0.0, 0.1, 0.3):
    r, b, f = retrieval_stats(drop)
    print(f"edges dropped {drop:>4.0%}: retrieval complete {r:6.1%} | buffer ~{b:,.0f} tok vs full prompt ~{f:,.0f} tok ({f/b:.0f}x smaller)")
print("(paper: 100% retrieval on the intact graph; 93.1% / 81.9% with 10% / 30% of edges dropped)")

## 5. Paper results (from the released aggregate results)

In [ ]:
E = os.path.join(REPO, "experiments")
# every number in the paper, produced by code/repomem/recompute_all.py from the per-item outputs
N = json.load(open(os.path.join(E, "paper_numbers.json")))
print("Main set (well-posed): all / in-window / beyond-window / in-window buckets 0-512..20K+")
for k, v in N["main"].items():
    print(f"  {k:11} {v['all']:5.1f} {v['in_window']:6.1f} {v['beyond']:6.1f}  {v['buckets']}")
S = N["scaling"]
print(f"\nSize scaling on the common subset that fits every model (n = {S['n']}):")
print(f"{'model':6} {'baseline':>9} {'oracle':>7} {'gap 2K-8K':>10} {'gap 8K-20K':>11} {'s/query':>8}")
for m in ("E2B", "E4B", "12B", "31B"):
    d = S[m]
    print(f"{m:6} {d['base']:9.1f} {d['oracle']:7.1f} {d['gap_2k8k']:10.1f} {d['gap_8k20k']:11.1f} {d['sec']:8.2f}")
print(f"E2B + PFK-agent: {S['E2B PFK']['all']} ({S['E2B PFK']['sec']} s/query)")
print("\nPFK-agent variants (E4B unless noted): all / in-window / beyond-window")
for k, v in N["ablations"].items():
    print(f"  {k:22} {v['all']:5.1f} {v['in_window']:6.1f} {v['beyond']:6.1f}")
A = N["seeds"]["agg"]
print("\nThree regenerated instances, mean±std: buckets 0-512, 512-2K, 2K-8K, 8K-20K, beyond, all")
for tag in ("base", "pfk", "oracle"):
    print(f"  {tag:7}", "  ".join(f"{m:.1f}±{s:.1f}" for m, s in A[tag]))

In [ ]:
from IPython.display import Image, display
for f in ("repomem_distance.png", "repomem_scaling.png", "repomem_seeds.png"):
    display(Image(os.path.join(REPO, "figures", f), width=820))

## 6. RepoMem-OSS: rebuild the open benchmark from source and verify it

RepoMem-OSS is built only from public repositories, so it is released as files in the GitHub repository. Here we rebuild it from scratch: clone the six repositories at the pinned commits, build the code graphs with `build_graph.py`, and regenerate the items. We then check that the result is **identical** to the released `repomem_oss.jsonl.gz`.

In [ ]:
import gzip, hashlib
OSS = os.path.join(REPO, "benchmark", "repomem-oss")
man = json.load(open(os.path.join(OSS, "MANIFEST.json")))
SRC, G2 = "/tmp/oss_src", "/tmp/oss_graphs"
os.makedirs(SRC, exist_ok=True); os.makedirs(G2, exist_ok=True)
for r in man["repos"]:
    d = os.path.join(SRC, r["name"])
    if not os.path.exists(d):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", r["tag"], r["github"] + ".git", d], check=True)
    sha = subprocess.check_output(["git", "-C", d, "rev-parse", "HEAD"]).decode().strip()
    assert sha == r["commit"], (r["name"], sha)
    subprocess.run([sys.executable, os.path.join(CODE, "build_graph.py"), "--repo", d,
                    "--name", f"{r['name']}_{r['tag']}", "--out", G2], check=True)
OUT = "/tmp/repomem_oss_rebuilt.jsonl"
subprocess.run([sys.executable, os.path.join(CODE, "generate.py"), "--graphs", G2, "--out", OUT,
                "--per-repo", "260", "--rounds", "14", "--seed", "1"], check=True, stdout=subprocess.DEVNULL)
rebuilt = open(OUT, "rb").read()
released = gzip.open(os.path.join(OSS, "repomem_oss.jsonl.gz")).read()
print("items rebuilt:", rebuilt.count(b"\n"), "| released:", released.count(b"\n"))
print("byte-identical to the released benchmark:", hashlib.sha256(rebuilt).hexdigest() == hashlib.sha256(released).hexdigest())
X = json.load(open(os.path.join(E, "ext_numbers.json")))["models"]
print("\nRepoMem-OSS single-hop accuracy by distance (0-512, 512-2K, 2K-8K, 8K-20K) and overall:")
for m in ("e2b_full", "e2b_pfk", "e2b_compact", "e4b_full", "e4b_pfk", "e4b_compact"):
    h = X[m]["hop"]
    print(f"  {m:12} {h['single']['buckets']}  all {h['single']['all']}  | multi-hop {h['multi']['all']}")

## 7. (Optional) Evaluate a Gemma 4 model

Requires a GPU, vLLM, and a Gemma 4 checkpoint (e.g. `gemma-4-e4b-it-qat-w4a16-ct`). Set `RUN_MODEL = True`.
The paper used vLLM 0.33.0 on 2× RTX 4090 (`scripts/serve_model.sh`), greedy decoding, thinking off.

In [ ]:
RUN_MODEL = False
MODEL_PATH = "/kaggle/input/gemma-4/transformers/gemma-4-e4b-it-qat-w4a16-ct/1"   # adjust to the attached model
if RUN_MODEL:
    srv = subprocess.Popen(["vllm", "serve", MODEL_PATH, "--served-model-name", "gemma",
                            "--max-model-len", "32768", "--port", "8000"])
    # wait for http://127.0.0.1:8000/v1/models, then:
    for cmd in (["evaluate.py", "--tasks", TASKS, "--model", "gemma", "--out", "base.jsonl", "--limit", "200"],
                ["pfk_agent.py", "--tasks", TASKS, "--graphs", GRAPHS, "--model", "gemma", "--out", "pfk.jsonl", "--limit", "200"]):
        subprocess.run([sys.executable, os.path.join(CODE, cmd[0])] + cmd[1:], check=True)